# 🔎 Étape 3 : Analyse Exploratoire des Données (EDA) (Squelette Étudiant)

Cette étape correspond au troisième chapitre du cours. L'objectif est d'explorer et de résumer les propriétés statistiques fondamentales du dataset **Automobile** et de réaliser du **Feature Engineering** pour enrichir les modèles.

### 1. Préparation de l'environnement

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sys.path.append(os.path.abspath('..'))

print("Librairies importées pour l'EDA !")

Librairies importées pour l'EDA !


### 2. Chargement des données nettoyées
**Rappel des modifications appliquées (Le "Clean Data") :**
Ce jeu de données est considéré comme "propre" car il a subi les transformations suivantes pour être parfaitement exploitable par nos algorithmes :
1. **Imputation :** Les valeurs manquantes de la colonne `horsepower` ont été remplacées par la moyenne.
2. **Encodage :** La variable textuelle `origin` a été transformée en valeurs numériques via `LabelEncoder` (ex: 0 pour Europe, 1 pour Japon, 2 pour USA).
3. **Nettoyage :** La colonne textuelle `name` a été supprimée en raison de sa trop forte cardinalité (ce qui évite le surapprentissage).

In [2]:
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

print(f"Shape : {df.shape}")
df.head()

Shape : (398, 8)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,2
1,15.0,8,350.0,165.0,3693,11.5,70,2
2,18.0,8,318.0,150.0,3436,11.0,70,2
3,16.0,8,304.0,150.0,3433,12.0,70,2
4,17.0,8,302.0,140.0,3449,10.5,70,2


### 3. Statistiques Descriptives

Résumés statistiques globaux et agrégations par groupe d'origine pour les variables clés du dataset.

In [4]:
print("=== Statistiques descriptives globales ===")
df.describe()

=== Statistiques descriptives globales ===


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000
mean,23.514573,5.454774,193.425879,104.304020,2970.424623,15.568090,76.010050,1.449749
std,7.815984,1.701004,104.269838,38.222625,846.841774,2.757689,3.697627,0.775076
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,0.000000
25%,17.500000,4.000000,104.250000,76.000000,2223.750000,13.825000,73.000000,1.000000
50%,23.000000,4.000000,148.500000,93.500000,2803.500000,15.500000,76.000000,2.000000
75%,29.000000,8.000000,262.000000,125.000000,3608.000000,17.175000,79.000000,2.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000,2.000000


Avant de croiser les variables, il faut connaître le comportement individuel de chaque colonne. <br>
La fonction `.describe()` nous donne une vision mathématique globale (moyenne, écart-type, quartiles) pour vérifier la dispersion de nos données.

**Interprétation des résultats :**
* La consommation (`mpg`) a une moyenne de ~23.5, mais le grand écart entre le minimum (9.0) et le maximum (46.6) prouve que notre jeu de données contient des profils de véhicules extrêmement variés.
* Le poids (`weight`) varie de 1613 à 5140 lbs. Ces immenses différences d'échelles justifient l'importance d'avoir créé notre ratio "puissance/poids" lors du Feature Engineering, pour remettre ces véhicules sur un pied d'égalité.

In [5]:
print("=== Moyennes par origine (0=europe, 1=japan, 2=usa) ===")
df.groupby('origin')[['mpg', 'horsepower', 'weight', 'acceleration']].mean().round(2)

=== Moyennes par origine (0=europe, 1=japan, 2=usa) ===


,mpg,horsepower,weight,acceleration
origin,,,,
0,27.89,80.93,2423.30,16.79
1,30.45,79.84,2221.23,16.17
2,20.08,118.64,3361.93,15.03


### 4. Ingénierie de variables (Feature Engineering)

Création de deux nouvelles variables dérivées :
- **power_to_weight** : rapport puissance/poids, indicateur d'efficacité mécanique
- **mpg_category** : discrétisation de la consommation en 3 tertiles (faible / moyenne / élevée)

In [6]:
df_feat = df.copy()

# Rapport puissance / poids
df_feat['power_to_weight'] = df_feat['horsepower'] / df_feat['weight']

# Catégorisation de la consommation en 3 tertiles
df_feat['mpg_category'] = pd.qcut(
    df_feat['mpg'],
    q=3,
    labels=['faible', 'moyenne', 'élevée']
)

print(f"Shape après feature engineering : {df_feat.shape}")
print(f"Nouvelles colonnes : power_to_weight, mpg_category")
df_feat[['mpg', 'horsepower', 'weight', 'power_to_weight', 'mpg_category']].head()

Shape après feature engineering : (398, 10)
Nouvelles colonnes : power_to_weight, mpg_category


,mpg,horsepower,weight,power_to_weight,mpg_category
0,18.0,130.0,3504,0.037100,faible
1,15.0,165.0,3693,0.044679,faible
2,18.0,150.0,3436,0.043655,faible
3,16.0,150.0,3433,0.043694,faible
4,17.0,140.0,3449,0.040591,faible


### 5. Analyse des Corrélations

Matrice de corrélation de Pearson sur toutes les colonnes numériques (y compris `power_to_weight`), puis pairplot des variables clés coloré par origine.

In [7]:
# Sélection des colonnes numériques incluant power_to_weight
numeric_cols = ['mpg', 'cylinders', 'displacement', 'horsepower',
                'weight', 'acceleration', 'model_year', 'origin', 'power_to_weight']

corr_matrix = df_feat[numeric_cols].corr(method='pearson')

print("=== Matrice de corrélation de Pearson ===")
print(corr_matrix.round(3))

=== Matrice de corrélation de Pearson ===
                   mpg  cylinders  displacement  horsepower  weight  \
mpg              1.000     -0.775        -0.804      -0.773  -0.832   
cylinders       -0.775      1.000         0.951       0.841   0.896   
displacement    -0.804      0.951         1.000       0.896   0.933   
horsepower      -0.773      0.841         0.896       1.000   0.862   
weight          -0.832      0.896         0.933       0.862   1.000   
acceleration     0.420     -0.505        -0.544      -0.687  -0.417   
model_year       0.579     -0.349        -0.370      -0.414  -0.307   
origin          -0.483      0.551         0.591       0.443   0.521   
power_to_weight -0.244      0.246         0.283       0.599   0.130   

                 acceleration  model_year  origin  power_to_weight  
mpg                     0.420       0.579  -0.483           -0.244  
cylinders              -0.505      -0.349   0.551            0.246  
displacement           -0.544      -0.37

La matrice de corrélation (méthode de Pearson) est l'outil central de l'EDA. Elle chiffre de -1 à 1 la force de la relation linéaire entre deux variables. Cela nous permet de sélectionner les variables prédictives les plus fortes pour l'algorithme et d'éliminer celles qui font doublon.

**Interprétation des résultats (Corrélation vs Causalité) :**
* **Corrélations négatives fortes :** Le poids (`weight`), la cylindrée (`displacement`) et la puissance (`horsepower`) sont fortement corrélés négativement avec la consommation (`mpg`). Mathématiquement, plus la voiture est lourde, moins elle est économique.
* **⚠️ Attention à la causalité :** un chiffre n'explique pas la physique. Ce n'est pas le simple fait d'ajouter un cylindre qui fait baisser le `mpg`. Un gros moteur implique un châssis plus lourd pour le soutenir, ce qui demande plus d'énergie pour avancer. La matrice montre les liens mathématiques, mais c'est notre compréhension métier qui valide la logique causale !

In [2]:
import pandas as pd
import plotly.express as px
import os

# 1. Chargement des données (Pour éviter le "NameError")
df = pd.read_csv('../data/processed/cleaned_data_sample.csv')

# Astuce : On crée une colonne temporaire pour avoir de vrais noms dans la légende
origin_map = {0: 'Europe', 1: 'Japan', 2: 'USA'}
df['Nom_Origine'] = df['origin'].map(origin_map)

# 2. Création de la matrice de nuages de points (SPLOM) avec Plotly Express
fig = px.scatter_matrix(
    df,
    dimensions=['mpg', 'horsepower', 'weight', 'acceleration'], # Les colonnes à croiser
    color='Nom_Origine',                                        # L'équivalent de hue='origin'
    color_discrete_sequence=['#3b82f6', '#ef4444', '#22c55e'],  # Vos couleurs (Bleu, Rouge, Vert)
    title="<b>Pairplot (Matrice de Nuages de Points) — Automobile Dataset</b>"
)

# 3. Mise en page globale
fig.update_layout(
    template="plotly_white",      # Votre thème sombre pour le dashboard
    height=800,                  # On le fait bien grand pour qu'il soit lisible
    width=900,
    hovermode="closest"
)

# On peaufine l'apparence des points (transparence et contour blanc)
fig.update_traces(
    diagonal_visible=False,      # Désactive la diagonale qui n'est pas très utile ici
    marker=dict(size=6, opacity=0.7, line=dict(width=0.5, color='white'))
)

# 4. Sauvegarde en format Web Interactif (.html) au lieu du .png
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
html_path = os.path.join(output_dir, 'pairplot_splom.html')
fig.write_html(html_path)

print(f"Graphique interactif sauvegardé : {html_path}")

# Affichage interactif dans le Notebook
fig.show()

Graphique interactif sauvegardé : ../data/processed\pairplot_splom.html


Une matrice de corrélation ne donne que des chiffres, or un chiffre peut cacher des distributions très différentes. Le `pairplot` trace toutes les variables les unes contre les autres pour confirmer visuellement nos corrélations et repérer des "clusters" (groupes) grâce à un code couleur basé sur l'origine du véhicule (`hue='origin'`).

**Ce que le Pairplot nous révèle (Conclusion de l'EDA) :**
1. **Des Clusters Géographiques Évidents :** 
   * Les véhicules Américains (`usa`) sont massivement concentrés dans les zones de haut poids, forte puissance et faible `mpg` (consommation élevée).
   * Les véhicules Japonais et Européens se regroupent presque exclusivement dans la zone inverse : véhicules légers et très économiques.
2. **Des formes non linéaires :** Si l'on regarde le croisement entre `mpg` et `weight` (ou `horsepower`), on remarque que la relation ne forme pas une ligne droite parfaite, mais plutôt une courbe. 
   * *Conséquence pour le ML :* Cela indique qu'un simple algorithme de "Régression Linéaire" classique aura peut-être du mal à tracer une ligne droite parfaite ici. Il faudra potentiellement appliquer une transformation logarithmique sur nos données pour "redresser" la courbe, ou utiliser des modèles plus complexes (comme les Arbres de Décision).

**Bilan :** Nos données sont maintenant parfaitement comprises, nettoyées et justifiées. Nous sommes prêts à passer à la phase de Machine Learning !